# Wafer Defect Detection Model Inference

This notebook demonstrates how to use the wafer defect detection model for inference.

Two approaches are shown:
1. **PyTorch model (.pth)**
2. **ONNX model (.onnx)** - Recommended for production

## Setup Instructions

### 1. Create a Virtual Environment (Recommended)
It's highly recommended to create a virtual environment to avoid conflicts with other packages:

**Windows:**
```bash
python -m venv venv
venv\Scripts\activate
```

**macOS/Linux:**
```bash
python -m venv venv
source venv/bin/activate
```

### 2. Install Dependencies
Once your virtual environment is activated, install all dependencies from `requirements.txt`:
```bash
pip install -r requirements.txt
```

### 3. Verify Installation
You can verify the installation by checking the versions:
```bash
pip list
```

## 1. Import Libraries

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import onnxruntime as ort

# Define class labels
CLASS_LABELS = [
    'Bridge', 'Clean', 'Etch', 'Open', 
    'Other', 'Particle', 'Scratch', 'Thermal'
]

## 2. PyTorch Model Functions

In [ ]:
def load_pytorch_model(model_path='modelv1.pth'):
    """
    Load the PyTorch model for inference.
    
    Args:
        model_path: Path to the .pth model file
        
    Returns:
        model: Loaded PyTorch model in evaluation mode
    """
    from torchvision import models
    
    # Load MobileNetV2 architecture
    model = models.mobilenet_v2(pretrained=False)
    
    # Modify the classifier for 8 classes
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 8)
    
    # Load trained weights
    model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
    model.eval()
    
    return model

In [ ]:
def preprocess_image_pytorch(image_path):
    """
    Preprocess image for PyTorch model.
    
    Args:
        image_path: Path to the input image
        
    Returns:
        tensor: Preprocessed image tensor
    """
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),  # Convert grayscale to 3-channel
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    image = Image.open(image_path)
    image_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    
    return image_tensor

In [ ]:
def predict_pytorch(model, image_path):
    """
    Make prediction using PyTorch model.
    
    Args:
        model: Loaded PyTorch model
        image_path: Path to the input image
        
    Returns:
        prediction: Dictionary with predicted class and confidence
    """
    # Preprocess image
    image_tensor = preprocess_image_pytorch(image_path)
    
    # Make prediction
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    predicted_class = CLASS_LABELS[predicted.item()]
    confidence_score = confidence.item()
    
    # Get all class probabilities
    all_probabilities = {
        CLASS_LABELS[i]: probabilities[0][i].item() 
        for i in range(len(CLASS_LABELS))
    }
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence_score,
        'all_probabilities': all_probabilities
    }

## 3. ONNX Model Functions (Recommended)

In [ ]:
def load_onnx_model(model_path='modelv1.onnx'):
    """
    Load the ONNX model for inference.
    
    Args:
        model_path: Path to the .onnx model file
        
    Returns:
        session: ONNX Runtime inference session
    """
    session = ort.InferenceSession(model_path)
    return session

In [ ]:
def preprocess_image_onnx(image_path):
    """
    Preprocess image for ONNX model.
    
    Args:
        image_path: Path to the input image
        
    Returns:
        numpy array: Preprocessed image array
    """
    # Load and preprocess image
    image = Image.open(image_path)
    
    # Convert grayscale to RGB
    if image.mode != 'RGB':
        image = image.convert('RGB')
    
    # Resize to 224x224
    image = image.resize((224, 224))
    
    # Convert to numpy array and normalize
    image_array = np.array(image).astype(np.float32) / 255.0
    
    # Apply ImageNet normalization
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    image_array = (image_array - mean) / std
    
    # Transpose to CHW format (channels, height, width)
    image_array = np.transpose(image_array, (2, 0, 1))
    
    # Add batch dimension
    image_array = np.expand_dims(image_array, axis=0)
    
    return image_array

In [ ]:
def predict_onnx(session, image_path):
    """
    Make prediction using ONNX model.
    
    Args:
        session: ONNX Runtime inference session
        image_path: Path to the input image
        
    Returns:
        prediction: Dictionary with predicted class and confidence
    """
    # Preprocess image
    image_array = preprocess_image_onnx(image_path)
    
    # Get input name
    input_name = session.get_inputs()[0].name
    
    # Run inference
    outputs = session.run(None, {input_name: image_array})
    
    # Apply softmax to get probabilities
    logits = outputs[0][0]
    exp_logits = np.exp(logits - np.max(logits))  # Subtract max for numerical stability
    probabilities = exp_logits / np.sum(exp_logits)
    
    # Get predicted class
    predicted_idx = np.argmax(probabilities)
    predicted_class = CLASS_LABELS[predicted_idx]
    confidence_score = probabilities[predicted_idx]
    
    # Get all class probabilities
    all_probabilities = {
        CLASS_LABELS[i]: float(probabilities[i]) 
        for i in range(len(CLASS_LABELS))
    }
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence_score,
        'all_probabilities': all_probabilities
    }

## 4. Batch Processing

In [ ]:
def predict_batch_onnx(session, image_paths):
    """
    Process multiple images in batch using ONNX model.
    
    Args:
        session: ONNX Runtime inference session
        image_paths: List of image file paths
        
    Returns:
        results: List of prediction dictionaries
    """
    results = []
    
    for image_path in image_paths:
        try:
            prediction = predict_onnx(session, image_path)
            prediction['image_path'] = image_path
            results.append(prediction)
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            results.append({
                'image_path': image_path,
                'error': str(e)
            })
    
    return results

## 5. Example Usage

### Method 1: PyTorch Inference

In [ ]:
# Example image path (replace with your actual image)
image_path = '../main/test/Particle/particle_001.jpg'

print("=" * 70)
print("PyTorch Model Inference")
print("=" * 70)

try:
    # Load model
    print("Loading PyTorch model...")
    pytorch_model = load_pytorch_model('modelv1.pth')
    
    # Make prediction
    print(f"Processing image: {image_path}")
    result = predict_pytorch(pytorch_model, image_path)
    
    # Display results
    print(f"\nPredicted Class: {result['predicted_class']}")
    print(f"Confidence: {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")
    print("\nAll Class Probabilities:")
    for class_name, prob in sorted(result['all_probabilities'].items(), 
                                   key=lambda x: x[1], reverse=True):
        print(f"  {class_name:12s}: {prob:.4f} ({prob*100:.2f}%)")

except Exception as e:
    print(f"PyTorch inference failed: {str(e)}")

### Method 2: ONNX Inference (Recommended for Production)

In [ ]:
print("=" * 70)
print("ONNX Model Inference (Recommended)")
print("=" * 70)

try:
    # Load model
    print("Loading ONNX model...")
    onnx_session = load_onnx_model('modelv1.onnx')
    
    # Make prediction
    print(f"Processing image: {image_path}")
    result = predict_onnx(onnx_session, image_path)
    
    # Display results
    print(f"\nPredicted Class: {result['predicted_class']}")
    print(f"Confidence: {result['confidence']:.4f} ({result['confidence']*100:.2f}%)")
    print("\nAll Class Probabilities:")
    for class_name, prob in sorted(result['all_probabilities'].items(), 
                                   key=lambda x: x[1], reverse=True):
        print(f"  {class_name:12s}: {prob:.4f} ({prob*100:.2f}%)")

except Exception as e:
    print(f"ONNX inference failed: {str(e)}")

### Method 3: Batch Processing Example

In [ ]:
print("=" * 70)
print("Batch Processing Example")
print("=" * 70)

try:
    # Example: Process multiple images
    image_paths = [
        '../main/test/Particle/particle_001.jpg',
        '../main/test/Scratch/scratch_001.jpg',
        '../main/test/Clean/clean_001.jpg',
    ]
    
    onnx_session = load_onnx_model('modelv1.onnx')
    results = predict_batch_onnx(onnx_session, image_paths)
    
    print(f"\nProcessed {len(results)} images:")
    for i, result in enumerate(results, 1):
        if 'error' not in result:
            print(f"\n{i}. {result['image_path']}")
            print(f"   → {result['predicted_class']} "
                  f"({result['confidence']*100:.2f}% confidence)")

except Exception as e:
    print(f"Batch processing failed: {str(e)}")

print("\n" + "=" * 70)